# Potential Talents — Reproducible Candidate Ranking

This notebook is the primary reviewer-facing executable artifact for the Potential Talents project. It reconstructs the candidate population from the immutable source data, applies transparent occupational HR rules, derives semantic relevance from pretrained Word2Vec embeddings, constructs the analytical target, evaluates PCA–Ridge with repeated nested cross-validation, audits management feedback, and produces the final reviewer-facing outputs.

The workflow is deliberately ordered so that empirical results are discovered before they are interpreted. Manual relevance grades are not used to define the target, eligibility rules, model, or final ranking.


# Execution and analytical contract

The notebook should run from the repository root with **Restart Kernel → Run All** after the required Python dependencies and NLPL Model 40 embedding archive are available.

The locked analytical target is

$$
G_i=\frac{H_i+W_i}{2},
$$

where $H$ is transparent occupational HR relevance and $W$ is semantic relevance to two HR-search queries. NDCG is the primary ranking metric. Raw model predictions are authoritative for ranking; clipping is presentation-only.

The production ranking is frozen before management feedback. Human feedback is evaluated as a governance and sensitivity experiment rather than silently redefining the analytical model.


# Set up the environment

The setup below resolves the repository root, imports the project modules, records the authoritative model specification, fixes the random seed, and captures a lightweight environment fingerprint. Analytical settings are supplied explicitly to modelling functions rather than hidden inside notebook cells.


In [ ]:
# Set up imports, paths, configuration and reproducibility

from pathlib import Path
import importlib.metadata as metadata
import json
import platform

import numpy as np
import pandas as pd
from IPython.display import Image, display

from src import (
    config,
    data_pipeline,
    embeddings,
    feedback,
    io_utils,
    modeling,
    presentation,
    ranking,
    validation,
)
from src.hr_rules import (
    ADJACENT_PATTERNS,
    DIRECT_PATTERNS,
    apply_hr_rules,
)

ROOT = config.project_root()
model_spec = config.get_model_spec()
np.random.seed(model_spec["seed"])

required_project_dirs = [
    ROOT / "src",
    ROOT / "data",
    ROOT / "models",
]

for directory in required_project_dirs:
    if not directory.exists():
        raise FileNotFoundError(
            f"Required project directory not found: {directory}"
        )

package_names = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "matplotlib",
]

environment_fingerprint = {
    package: metadata.version(package)
    for package in package_names
}
environment_fingerprint["python"] = platform.python_version()

print("Repository root:", ROOT)
print("Model specification:")
display(
    pd.DataFrame(
        {
            "setting": list(model_spec.keys()),
            "value": [
                (
                    value.tolist()
                    if isinstance(value, np.ndarray)
                    else value
                )
                for value in model_spec.values()
            ],
        }
    )
)
print("Environment:")
display(pd.Series(environment_fingerprint, name="version"))


# Verify the setup

The model specification printed above is the single notebook-visible configuration used throughout PCA, Ridge tuning and repeated nested cross-validation. In particular, the production design uses 95% PCA variance retention with the full SVD solver, five outer folds repeated ten times, four inner folds, a 15-value Ridge alpha grid, seed 42, MSE tuning, and the locked NDCG cutoffs.

No project-specific empirical counts have been assumed at this stage.


# Load and inspect the immutable source data

The first analytical step is source validation. The raw CSV is loaded without assuming its row count, its required schema is checked, and a SHA-256 hash is recorded before any transformation is applied.


In [ ]:
# Load the raw candidate data and validate its schema

raw_path = ROOT / "data" / "raw" / "potential-talents.csv"

if not raw_path.exists():
    raise FileNotFoundError(
        f"Raw candidate file not found: {raw_path}"
    )

raw_sha256 = io_utils.sha256_file(raw_path)
raw_df = data_pipeline.load_raw(raw_path)

required_raw_columns = {
    "id",
    "job_title",
    "location",
    "connection",
    "fit",
}

missing_columns = required_raw_columns.difference(
    raw_df.columns
)

if missing_columns:
    raise ValueError(
        "Raw data are missing required columns: "
        f"{sorted(missing_columns)}"
    )

if raw_df["id"].duplicated().any():
    raise ValueError(
        "Raw candidate IDs must be unique."
    )

raw_summary = pd.DataFrame(
    {
        "measure": [
            "rows",
            "columns",
            "unique exact job titles",
            "missing fit values",
        ],
        "value": [
            len(raw_df),
            raw_df.shape[1],
            raw_df["job_title"].nunique(
                dropna=True
            ),
            int(raw_df["fit"].isna().sum()),
        ],
    }
)

print("Raw SHA-256:", raw_sha256)
display(raw_summary)
display(raw_df.head())


# Interpret the raw-source validation

The source inspection establishes the actual dataset dimensions and confirms that the supplied `fit` field is not an observed relevance target. The project therefore constructs its own analytical relevance target later in the workflow.

The raw-file hash recorded above anchors the analysis to an immutable source version and is carried into the final reproducibility manifest.


# Reduce repeated profiles and remove invalid source records

Repeated exact job titles represent duplicate profile text rather than independent analytical observations. The next step collapses exact-title groups to their lowest source candidate ID, applies the two explicit content-validity rules, and then deduplicates normalized job titles using the same deterministic lowest-ID rule.

All resulting counts are discovered from execution rather than supplied as expected inputs.


In [ ]:
# Construct the valid unique candidate population

exact_title_profiles = (
    data_pipeline.exact_title_universe(
        raw_df
    )
)

valid_title_profiles, exclusion_audit = (
    data_pipeline.remove_invalid(
        exact_title_profiles
    )
)

clean_candidates, dedup_audit = (
    data_pipeline.normalized_dedup(
        valid_title_profiles,
        return_audit=True,
    )
)

population_reduction = pd.DataFrame(
    {
        "stage": [
            "Raw source rows",
            "Exact-title profiles",
            "Valid exact-title profiles",
            "Valid normalized-title profiles",
        ],
        "count": [
            len(raw_df),
            len(exact_title_profiles),
            len(valid_title_profiles),
            len(clean_candidates),
        ],
    }
)

display(population_reduction)

print("Content exclusions:")
display(exclusion_audit)

print("Normalized-title deduplication audit:")
display(
    dedup_audit.loc[
        ~dedup_audit["kept"]
    ]
)


# Interpret the population reduction

The reduction separates data-quality decisions from later relevance modelling. Exact-title consolidation prevents duplicated profile text from receiving repeated weight, the invalid-content rules remove records that are not genuine candidate roles, and normalized-title deduplication resolves textual variants deterministically.

The resulting valid unique population is the universe used for both occupational filtering and the later full-population semantic stress test.


# Define transparent occupational HR relevance

Occupational relevance $H$ is assigned by explicit rule lists with three possible values:

$$
H =
\begin{cases}
1.0 & \text{direct HR occupational evidence}\\
0.5 & \text{adjacent people-development evidence}\\
0.0 & \text{no explicit HR occupational evidence}
\end{cases}
$$

These rules determine eligibility for the production modelling population. Semantic similarity is not allowed to override occupational eligibility.


In [ ]:
# Apply the occupational HR rules

hr_rule_audit, hr34_from_rules = apply_hr_rules(
    clean_candidates
)

hr_class_summary = (
    hr_rule_audit
    .groupby(
        ["hr_class", "H"],
        dropna=False,
    )
    .size()
    .reset_index(
        name="candidate_count"
    )
)

hr_candidates = (
    hr_rule_audit.loc[
        hr_rule_audit[
            "hr_class"
        ] != "irrelevant"
    ]
    .sort_values(
        "representative_id"
    )
    .reset_index(drop=True)
)

assert np.array_equal(
    hr_candidates[
        "representative_id"
    ].to_numpy(dtype=int),
    hr34_from_rules[
        "representative_id"
    ].to_numpy(dtype=int),
)

excluded_candidates = (
    hr_rule_audit.loc[
        hr_rule_audit[
            "hr_class"
        ] == "irrelevant"
    ]
    .sort_values(
        "representative_id"
    )
    .reset_index(drop=True)
)

hr_ruleset = {
    "direct_patterns":
        list(DIRECT_PATTERNS),
    "adjacent_patterns":
        list(ADJACENT_PATTERNS),
    "H_map":
        dict(config.H_MAP),
}

hr_ruleset_sha256 = (
    io_utils.canonical_json_hash(
        hr_ruleset
    )
)

display(hr_class_summary)

print("Retained HR modelling population:")
display(
    hr_candidates[
        [
            "representative_id",
            "job_title",
            "hr_class",
            "H",
            "hr_rule_id",
            "hr_reason",
        ]
    ]
)

print("Excluded occupational profiles:")
display(
    excluded_candidates[
        [
            "representative_id",
            "job_title",
            "hr_reason",
        ]
    ]
)


# Interpret the HR relevance rules

Execution identifies 33 direct HR profiles, one adjacent profile and 16 profiles with no explicit occupational HR evidence, producing the final HR34 modelling population.

The adjacent case receives $H=0.5$ because its people-development role is related to HR without being a direct HR title. The remaining retained candidates receive $H=1$. The excluded group is not described as semantically irrelevant; it is excluded because its job-title evidence does not establish occupational HR relevance.

This distinction between **eligibility** and **semantic similarity** is maintained throughout the notebook.


# Explore HR-title language before defining the search queries

A focused descriptive text EDA is performed on the HR34 job titles before the semantic queries are fixed. The EDA tokenizer is intentionally simple and separate from the later Word2Vec preprocessing: it lowercases titles, retains contiguous alphabetic tokens and removes only a small fixed stopword set.

Its purpose is descriptive query construction, not model feature engineering.


In [ ]:
# Summarize recurring words and phrases in HR34

eda_summary = (
    data_pipeline.text_frequency_summary(
        hr_candidates["job_title"]
    )
)

print("EDA collection summary:")
display(eda_summary["summary"])

print("Most frequent words:")
display(
    eda_summary["unigrams"].head(20)
)

print("Most frequent bigrams:")
display(
    eda_summary["bigrams"].head(20)
)

QUERIES = [
    "aspiring human resources",
    "seeking human resources",
]

print("Locked semantic queries:", QUERIES)


# Interpret the title-language EDA and define the semantic queries

The HR34 titles are dominated by explicit human-resources terminology, while recurring intent terms such as **aspiring** and **seeking** identify candidates signalling a desired move into HR. This supports two complementary search formulations:

- `aspiring human resources`
- `seeking human resources`

The two queries are therefore fixed here, after the descriptive EDA and before embedding-based scoring.

These phrases are not treated as labels. They are semantic reference points used to measure how closely each candidate title aligns with the HR-search language.


# Represent all valid profiles with NLPL Model 40 Word2Vec

The semantic representation uses **NLPL Model 40: English CoNLL17 Word2Vec, 100 dimensions**.

The repository uses the **actual Model 40 binary (`models/model.bin`) as the authoritative embedding source**. No compact vector cache, precomputed candidate embedding file, or cache fallback is used.

To avoid using different vocabularies for retained and excluded candidates, the required token set is constructed once from **all 50 valid candidate titles plus the two queries**. The binary model is streamed once and only the required vectors are retained in memory. The same extracted vector dictionary is then used for the full-50 embeddings, the HR34 subset and both query embeddings.

This also allows the occupational filter to be stress-tested using exactly the same semantic representation used downstream.


In [ ]:
# Build the shared Model 40 representation and inspect coverage

MODEL_BIN_PATH = (
    ROOT
    / "models"
    / "model.bin"
)

MODEL_META_PATH = (
    ROOT
    / "models"
    / "meta.json"
)

embedding_texts = (
    clean_candidates["job_title"].tolist()
    + QUERIES
)

required_vocabulary = (
    embeddings.required_tokens(
        embedding_texts
    )
)

model40_metadata = (
    embeddings.load_model40_metadata(
        MODEL_META_PATH
    )
)

vectors = (
    embeddings.extract_required_vectors_from_bin(
        MODEL_BIN_PATH,
        required_vocabulary,
    )
)

coverage_summary = (
    embeddings.model40_coverage(
        required_vocabulary,
        vectors,
    )
)

oov_required_tokens = (
    coverage_summary[
        "oov_required_tokens"
    ]
)

all50_embeddings, all50_diagnostics = (
    embeddings.embedding_matrix(
        clean_candidates["job_title"],
        vectors,
    )
)

query_embeddings, query_diagnostics = (
    embeddings.embedding_matrix(
        QUERIES,
        vectors,
    )
)

clean_id_to_row = {
    int(candidate_id): row
    for row, candidate_id
    in enumerate(
        clean_candidates[
            "representative_id"
        ].astype(int)
    )
}

hr_embedding_rows = np.array(
    [
        clean_id_to_row[int(candidate_id)]
        for candidate_id
        in hr_candidates[
            "representative_id"
        ]
    ],
    dtype=int,
)

candidate_embeddings = (
    all50_embeddings[
        hr_embedding_rows
    ]
)

candidate_diagnostics = [
    all50_diagnostics[row]
    for row in hr_embedding_rows
]

embedding_diagnostics = pd.DataFrame(
    all50_diagnostics
).assign(
    representative_id=(
        clean_candidates[
            "representative_id"
        ].to_numpy()
    )
)

query_diagnostics_table = pd.DataFrame(
    query_diagnostics
).assign(query=QUERIES)

print(
    "Embedding source:",
    MODEL_BIN_PATH,
)
print(
    "Model 40 ID:",
    model40_metadata["id"],
)
print(
    "Model 40 vocabulary:",
    model40_metadata[
        "vocabulary size"
    ],
)
print(
    "Model 40 dimensions:",
    model40_metadata[
        "dimensions"
    ],
)
print(
    "Required unique tokens:",
    coverage_summary[
        "required_token_count"
    ],
)
print(
    "Vectors found:",
    coverage_summary[
        "found_token_count"
    ],
)
print(
    "OOV required tokens:",
    oov_required_tokens,
)

print("Candidate embedding coverage:")
display(
    embedding_diagnostics[
        [
            "representative_id",
            "text",
            "token_count",
            "recognized_count",
            "coverage_ratio",
            "oov",
        ]
    ]
)

print("Query embedding coverage:")
display(query_diagnostics_table)

assert MODEL_BIN_PATH.exists()
assert MODEL_BIN_PATH.stat().st_size > 0
assert (
    model40_metadata["id"]
    == embeddings.EXPECTED_MODEL40_ID
)
assert (
    model40_metadata["vocabulary size"]
    == embeddings.EXPECTED_MODEL40_VOCAB
)
assert (
    model40_metadata["dimensions"]
    == embeddings.EXPECTED_DIM
)
assert (
    all50_embeddings.shape
    == (
        len(clean_candidates),
        embeddings.EXPECTED_DIM,
    )
)
assert (
    candidate_embeddings.shape
    == (
        len(hr_candidates),
        embeddings.EXPECTED_DIM,
    )
)
assert query_embeddings.shape == (
    len(QUERIES),
    embeddings.EXPECTED_DIM,
)


# Verify the semantic representation

The coverage tables above document which title tokens are represented by the **actual Model 40 binary** rather than assuming perfect vocabulary coverage. Every HR34 vector is taken directly from the full-50 embedding matrix, so retained and excluded profiles remain comparable under one representation.

The two query vectors are generated through the same preprocessing and mean-embedding procedure. Any out-of-vocabulary terms are therefore visible before similarity scores are calculated.

Because the required vectors are extracted directly from `models/model.bin` during execution, there is no compact-cache provenance problem and no hidden fallback path to precomputed embeddings.


In [ ]:
# Stress-test the 50-to-34 occupational filter with semantic similarity

all50_raw_cosine = (
    embeddings.cosine_matrix(
        all50_embeddings,
        query_embeddings,
    )
)

all50_scaled_query_scores, W50 = (
    embeddings.semantic_scores(
        all50_embeddings,
        query_embeddings,
    )
)

semantic_filter_audit = (
    clean_candidates[
        [
            "representative_id",
            "job_title",
        ]
    ]
    .copy()
)

semantic_filter_audit[
    "cosine_query_1"
] = all50_raw_cosine[:, 0]

semantic_filter_audit[
    "cosine_query_2"
] = all50_raw_cosine[:, 1]

semantic_filter_audit[
    "scaled_query_1"
] = all50_scaled_query_scores[:, 0]

semantic_filter_audit[
    "scaled_query_2"
] = all50_scaled_query_scores[:, 1]

semantic_filter_audit["W"] = W50

semantic_filter_audit = (
    semantic_filter_audit.merge(
        hr_rule_audit[
            [
                "representative_id",
                "hr_class",
                "H",
                "hr_rule_id",
                "hr_reason",
            ]
        ],
        on="representative_id",
        how="left",
        validate="one_to_one",
    )
)

semantic_filter_audit[
    "retained_for_modeling"
] = (
    semantic_filter_audit[
        "hr_class"
    ] != "irrelevant"
)

all50_ids = (
    semantic_filter_audit[
        "representative_id"
    ]
    .to_numpy(dtype=int)
)

all50_semantic_order = (
    ranking.deterministic_order(
        semantic_filter_audit[
            "W"
        ].to_numpy(dtype=float),
        all50_ids,
    )
)

semantic_ranks = np.empty(
    len(semantic_filter_audit),
    dtype=int,
)

semantic_ranks[
    all50_semantic_order
] = np.arange(
    1,
    len(semantic_filter_audit) + 1,
)

semantic_filter_audit[
    "semantic_rank_50"
] = semantic_ranks

removal_notes = {
    2:
        "Teaching / education role; no HR role or HR-seeking signal.",
    5:
        "Advisory / governance role; no explicit HR occupational signal.",
    11:
        "Student status only; no HR role or HR career intent stated.",
    80:
        "Engineering / information-systems role rather than HR.",
    85:
        "Brand / portfolio commercial role rather than HR.",
    86:
        "Information-systems / programming role rather than HR.",
    87:
        "Biology education profile with no HR role or HR intent stated.",
    90:
        "Academic laboratory research role rather than HR.",
    91:
        "Generic institutional role with no explicit HR responsibility.",
    92:
        "Explicitly seeking Customer Service or Patient Care rather than HR.",
    93:
        "Admissions role; no evidence that the role concerns employee HR/recruitment.",
    95:
        "Student status only; no HR role or HR career intent stated.",
    96:
        "Business / retail-management profile with no HR role or HR intent stated.",
    98:
        "Generic student profile; insufficient evidence of HR relevance.",
    102:
        "Business-intelligence / analytics role rather than HR.",
    104:
        "Administration role with no explicit HR responsibility in the title.",
}

semantic_filter_audit[
    "removal_note"
] = (
    semantic_filter_audit[
        "representative_id"
    ].map(removal_notes)
)

excluded_semantic_audit = (
    semantic_filter_audit.loc[
        ~semantic_filter_audit[
            "retained_for_modeling"
        ]
    ]
    .sort_values(
        [
            "semantic_rank_50",
            "representative_id",
        ]
    )
    .reset_index(drop=True)
)

semantic_group_summary = (
    semantic_filter_audit
    .assign(
        population=np.where(
            semantic_filter_audit[
                "retained_for_modeling"
            ],
            "Retained HR population",
            "Occupationally excluded",
        )
    )
    .groupby("population")["W"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
        ]
    )
    .reset_index()
)

minimum_retained_W = float(
    semantic_filter_audit.loc[
        semantic_filter_audit[
            "retained_for_modeling"
        ],
        "W",
    ].min()
)

excluded_above_minimum_retained = int(
    (
        excluded_semantic_audit["W"]
        > minimum_retained_W
    ).sum()
)

first_excluded_rank = int(
    excluded_semantic_audit[
        "semantic_rank_50"
    ].min()
)

print("All occupational exclusions:")
display(
    excluded_semantic_audit[
        [
            "semantic_rank_50",
            "representative_id",
            "job_title",
            "cosine_query_1",
            "cosine_query_2",
            "W",
            "hr_reason",
            "removal_note",
        ]
    ]
)

print("Retained vs excluded W summary:")
display(semantic_group_summary)

print(
    "First excluded semantic rank:",
    first_excluded_rank,
)
print(
    "Excluded profiles above the "
    "minimum retained W:",
    excluded_above_minimum_retained,
)

overlap_window = (
    semantic_filter_audit
    .sort_values(
        "semantic_rank_50"
    )
    .loc[
        lambda frame:
            frame["semantic_rank_50"]
            .between(
                max(1, first_excluded_rank - 3),
                min(
                    len(frame),
                    first_excluded_rank + 6,
                ),
            )
    ]
)

print(
    "Semantic ranking around the first exclusion:"
)
display(
    overlap_window[
        [
            "semantic_rank_50",
            "representative_id",
            "job_title",
            "W",
            "retained_for_modeling",
        ]
    ]
)


# Interpret the semantic filter stress test

The semantic ranking broadly supports the occupational filter, but the retained and excluded distributions overlap. Some excluded profiles contain organizational, management, administration or employment language that produces meaningful semantic proximity to the HR queries without establishing occupational HR relevance.

That overlap is precisely why $W$ is not used as an eligibility threshold. A pure semantic cutoff could admit false positives and remove legitimate HR candidates whose titles are short or phrased differently.

The filter therefore intentionally favours occupational precision over recall. A transferable candidate with an unconventional title or unstated HR intent may be excluded, and this remains an explicit limitation.

Because the two queries were developed from the candidate-language EDA, this experiment is an internal consistency and stress test rather than independent external validation.


# Construct the analytical relevance target

For each HR candidate, semantic relevance is calculated from the same two query similarities:

$$
W_i =
\frac{1}{2}
\left[
\frac{s_{i1}+1}{2}
+
\frac{s_{i2}+1}{2}
\right],
$$

where $s_{i1}$ and $s_{i2}$ are raw cosine similarities.

The final analytical target is

$$
G_i=\frac{H_i+W_i}{2}.
$$

Manual relevance grades are not part of $H$, $W$, $G$, filtering, model training or the final ranking.


In [ ]:
# Calculate W and G for HR34 and construct the reference ranking

hr_ids = (
    hr_candidates[
        "representative_id"
    ]
    .to_numpy(dtype=int)
)

scaled_query_scores, W = (
    embeddings.semantic_scores(
        candidate_embeddings,
        query_embeddings,
    )
)

H = (
    hr_candidates["H"]
    .to_numpy(dtype=float)
)

G = (H + W) / 2.0

validation.require_finite(
    G,
    "TARGET_FINITE",
)

if not np.all(
    (G >= 0.0)
    & (G <= 1.0)
):
    raise ValueError(
        "G must remain in [0, 1]."
    )

hr_analysis = (
    hr_candidates[
        [
            "representative_id",
            "job_title",
            "hr_class",
            "H",
            "hr_rule_id",
            "hr_reason",
        ]
    ]
    .copy()
)

hr_analysis[
    "scaled_query_1"
] = scaled_query_scores[:, 0]

hr_analysis[
    "scaled_query_2"
] = scaled_query_scores[:, 1]

hr_analysis["W"] = W
hr_analysis["G"] = G

reference_order = (
    ranking.deterministic_order(
        G,
        hr_ids,
    )
)

reference_ranks = np.empty(
    len(hr_ids),
    dtype=int,
)

reference_ranks[
    reference_order
] = np.arange(
    1,
    len(hr_ids) + 1,
)

hr_analysis[
    "G_rank"
] = reference_ranks

X = np.asarray(
    candidate_embeddings,
    dtype=float,
)
y = np.asarray(
    G,
    dtype=float,
)
ids = hr_ids.copy()

print("Analytical target table:")
display(
    hr_analysis.sort_values(
        "G_rank"
    )
)

print(
    "G range:",
    float(G.min()),
    "to",
    float(G.max()),
)


# Model the analytical relevance target

The learned model receives only the 100-dimensional Word2Vec candidate representation $X$ and predicts $G$.

Within every training fold, PCA retains 95% of variance, StandardScaler standardizes the retained components, and Ridge regression is tuned over the central alpha grid. PCA, scaling and alpha selection are all fitted inside the appropriate training folds to prevent leakage.

The primary evaluation is ranking quality under repeated nested cross-validation, with regression and rank-correlation metrics retained as supporting diagnostics.


In [ ]:
# Tune and evaluate PCA-Ridge with repeated nested cross-validation

(
    cv_oof_predictions,
    cv_fold_audit,
    cv_repeat_metrics,
    oof_by_repeat,
    alpha_tuning_trace,
) = modeling.repeated_nested_cv(
    X=X,
    y=y,
    ids=ids,
    model_spec=model_spec,
)

expected_outer_fits = (
    model_spec["outer_splits"]
    * model_spec["outer_repeats"]
)

assert (
    len(cv_fold_audit)
    == expected_outer_fits
)

assert (
    len(cv_repeat_metrics)
    == model_spec["outer_repeats"]
)

assert (
    len(cv_oof_predictions)
    == len(ids)
    * model_spec["outer_repeats"]
)

assert (
    len(alpha_tuning_trace)
    == expected_outer_fits
    * len(model_spec["alpha_grid"])
)

metric_columns = [
    *[
        f"ndcg_at_{k}"
        for k
        in model_spec[
            "ndcg_cutoffs"
        ]
    ],
    "mae",
    "rmse",
    "spearman",
    "kendall",
]

cv_metric_summary = pd.DataFrame(
    {
        "metric": metric_columns,
        "mean": [
            float(
                cv_repeat_metrics[
                    metric
                ].mean()
            )
            for metric
            in metric_columns
        ],
        "sd": [
            float(
                cv_repeat_metrics[
                    metric
                ].std(ddof=1)
            )
            for metric
            in metric_columns
        ],
    }
)

print("Repeated nested-CV design:")
display(
    pd.Series(
        {
            "outer_splits":
                model_spec[
                    "outer_splits"
                ],
            "outer_repeats":
                model_spec[
                    "outer_repeats"
                ],
            "inner_splits":
                model_spec[
                    "inner_splits"
                ],
            "alpha_count":
                len(
                    model_spec[
                        "alpha_grid"
                    ]
                ),
            "pca_variance":
                model_spec[
                    "pca_variance"
                ],
            "tuning_objective":
                model_spec[
                    "tuning_objective"
                ],
        }
    )
)

print("Repeat-level metrics:")
display(cv_repeat_metrics)

print("Mean ± SD summary:")
display(cv_metric_summary)

print("Outer-fold audit:")
display(cv_fold_audit)

print("Alpha tuning trace:")
display(alpha_tuning_trace)


# Interpret the cross-validation results

The repeated out-of-fold results show consistently strong ranking performance across all locked NDCG cutoffs. The small between-repeat dispersion indicates that the ranking result is not being driven by one favourable train/test split.

MAE and RMSE describe numerical agreement with $G$, while Spearman and Kendall evaluate ordering across the complete population. These secondary metrics need not move identically with NDCG because NDCG places greater operational emphasis on getting the highest-relevance candidates near the top of the list.

The repeated nested design is the authoritative model-evaluation evidence; the later full-data fit is used only to produce the deployable ranking after model selection is complete.


In [ ]:
# Test whether the tuning objective changes the ranking

ndcg_diagnostic_spec = {
    **model_spec,
    "tuning_objective": "ndcg",
}

(
    ndcg_oof_predictions,
    ndcg_fold_audit,
    ndcg_repeat_metrics,
    ndcg_oof_by_repeat,
    ndcg_alpha_tuning_trace,
) = modeling.repeated_nested_cv_ndcg_diag(
    X=X,
    y=y,
    ids=ids,
    model_spec=ndcg_diagnostic_spec,
    k=10,
)

mse_mean_ndcg10 = float(
    cv_repeat_metrics[
        "ndcg_at_10"
    ].mean()
)

ndcg_tuned_mean_ndcg10 = float(
    ndcg_repeat_metrics[
        "ndcg_at_10"
    ].mean()
)

mse_mean_oof = np.vstack(
    [
        oof_by_repeat[key]
        for key
        in sorted(oof_by_repeat)
    ]
).mean(axis=0)

ndcg_mean_oof = np.vstack(
    [
        ndcg_oof_by_repeat[key]
        for key
        in sorted(
            ndcg_oof_by_repeat
        )
    ]
).mean(axis=0)

mse_top10 = set(
    ids[
        ranking.deterministic_order(
            mse_mean_oof,
            ids,
        )[:10]
    ]
)

ndcg_top10 = set(
    ids[
        ranking.deterministic_order(
            ndcg_mean_oof,
            ids,
        )[:10]
    ]
)

top10_membership_changes = (
    len(
        mse_top10.symmetric_difference(
            ndcg_top10
        )
    )
    // 2
)

tuning_objective_comparison = (
    pd.DataFrame(
        [
            {
                "strategy":
                    "MSE tuning",
                "mean_ndcg_at_10":
                    mse_mean_ndcg10,
                "top10_membership_changes_vs_mse":
                    0,
            },
            {
                "strategy":
                    "NDCG tuning",
                "mean_ndcg_at_10":
                    ndcg_tuned_mean_ndcg10,
                "top10_membership_changes_vs_mse":
                    top10_membership_changes,
            },
        ]
    )
)

ndcg_tuning_materiality = bool(
    abs(
        ndcg_tuned_mean_ndcg10
        - mse_mean_ndcg10
    ) >= 0.02
    or top10_membership_changes >= 2
)

display(tuning_objective_comparison)

print(
    "Material tuning-objective change:",
    ndcg_tuning_materiality,
)


# Interpret the tuning-objective sensitivity check

Direct inner-fold NDCG tuning produces little material change relative to the simpler MSE tuning strategy. The difference in NDCG@10 is small and top-candidate membership remains essentially stable.

MSE therefore remains the production tuning objective. It supplies a continuous optimization criterion, avoids adding a second ranking-specific tuning layer, and does not sacrifice meaningful ranking quality in this dataset.


In [ ]:
# Compare PCA-Ridge with the deterministic W-only baseline

W_baseline_scores = W.copy()

w_baseline_metrics = (
    ranking.evaluation_metrics(
        y_true=y,
        y_score=W_baseline_scores,
        cutoffs=model_spec[
            "ndcg_cutoffs"
        ],
    )
)

baseline_rows = []

for k in model_spec[
    "ndcg_cutoffs"
]:
    metric = f"ndcg_at_{k}"

    ridge_mean = float(
        cv_repeat_metrics[
            metric
        ].mean()
    )

    ridge_sd = float(
        cv_repeat_metrics[
            metric
        ].std(ddof=1)
    )

    baseline_rows.append(
        {
            "metric":
                f"NDCG@{k}",
            "W_only":
                w_baseline_metrics[
                    metric
                ],
            "Ridge_mean":
                ridge_mean,
            "Ridge_sd":
                ridge_sd,
            "Ridge_minus_W":
                ridge_mean
                - w_baseline_metrics[
                    metric
                ],
        }
    )

baseline_vs_ridge = pd.DataFrame(
    baseline_rows
)

display(baseline_vs_ridge)

print("W-only supporting metrics:")
display(
    pd.Series(
        {
            key: value
            for key, value
            in w_baseline_metrics.items()
            if not key.startswith(
                "ndcg_at_"
            )
        }
    )
)


# Interpret the deterministic baseline comparison

The deterministic $W$-only ranking is exceptionally strong and is not an independent external competitor to Ridge. $W$ is one half of the constructed target $G$, and $H=1$ for almost the entire HR34 population. Within that dominant subgroup,

$$
G_i=\frac{1+W_i}{2},
$$

so $G$ is a monotonic transformation of $W$.

The comparison is therefore a simplicity and structural baseline. Ridge demonstrates that the analytical relevance pattern can be recovered from the 100-dimensional candidate embeddings under a leakage-controlled learning pipeline; it is not expected to outperform a score already embedded directly in the target construction.


In [ ]:
# Fit the final pre-feedback PCA-Ridge model

production_alpha, production_tuning_trace = (
    modeling.select_alpha(
        X=X,
        y=y,
        model_spec=model_spec,
    )
)

production_model, production_predictions_raw = (
    modeling.fit_full(
        X=X,
        y=y,
        alpha=production_alpha,
        model_spec=model_spec,
    )
)

production_order = (
    ranking.deterministic_order(
        production_predictions_raw,
        ids,
    )
)

production_rank_lookup = np.empty(
    len(ids),
    dtype=int,
)

production_rank_lookup[
    production_order
] = np.arange(
    1,
    len(ids) + 1,
)

frozen_pre_feedback_ranking = (
    hr_analysis.copy()
)

frozen_pre_feedback_ranking[
    "prediction_raw"
] = production_predictions_raw

frozen_pre_feedback_ranking[
    "ridge_rank"
] = production_rank_lookup

frozen_pre_feedback_ranking = (
    frozen_pre_feedback_ranking
    .sort_values(
        "ridge_rank"
    )
    .reset_index(drop=True)
)

print(
    "Selected full-data Ridge alpha:",
    production_alpha,
)

print("Full-data tuning trace:")
display(production_tuning_trace)

print("Frozen pre-feedback ranking:")
display(frozen_pre_feedback_ranking)


# Freeze the automated ranking before management review

The full-data Ridge ranking above is frozen before any human feedback is introduced. It is the automated reference against which the management experiments are evaluated.

Management may reorder selected candidates, but the experiment preserves candidate IDs, the existing $G$-score multiset and the distinction between analytical scores and human preference. Review is examined cumulatively at approximately 10%, 20% and 50% of HR34.

Two different questions are kept separate:

1. **Fitted intervention:** can Ridge accommodate the supplied management ordering?
2. **Generalization:** does retraining on the adjusted target improve held-out ranking performance?

A management reorder is therefore evidence for governance and inspection, not automatic permission to replace the production model.


In [ ]:
# Define the recorded management review

management_source_top25_order = [
    82, 76, 3, 6, 66,
    10, 94, 1, 27, 73,
    100, 72, 67, 74, 78,
    81, 101, 71, 70, 7,
    79, 28, 99, 89, 88,
]

assert len(
    management_source_top25_order
) == 25

assert len(
    set(
        management_source_top25_order
    )
) == 25

assert set(
    management_source_top25_order
).issubset(set(ids.tolist()))

baseline_top17 = (
    frozen_pre_feedback_ranking
    .sort_values("ridge_rank")
    .head(17)[
        "representative_id"
    ]
    .astype(int)
    .tolist()
)

baseline_top4 = baseline_top17[:4]
baseline_top7 = baseline_top17[:7]

management_top17_order = (
    feedback.restrict_order(
        management_source_top25_order,
        baseline_top17,
    )
)

assert len(
    management_top17_order
) == len(baseline_top17)

assert set(
    management_top17_order
) == set(baseline_top17)

feedback_cohorts = {
    4: baseline_top4,
    7: baseline_top17[4:7],
    17: baseline_top17[7:17],
}

feedback_desired_orders = {
    cutoff:
        feedback.restrict_order(
            management_source_top25_order,
            cohort_ids,
        )
    for cutoff, cohort_ids
    in feedback_cohorts.items()
}

review_plan_rows = []

for cutoff, cohort_ids in (
    feedback_cohorts.items()
):
    desired_order = (
        feedback_desired_orders[
            cutoff
        ]
    )

    for candidate_id in cohort_ids:
        review_plan_rows.append(
            {
                "cumulative_review_k":
                    cutoff,
                "representative_id":
                    candidate_id,
                "automated_position_in_cohort":
                    cohort_ids.index(
                        candidate_id
                    ) + 1,
                "management_position_in_cohort":
                    desired_order.index(
                        candidate_id
                    ) + 1,
            }
        )

management_review_plan = (
    pd.DataFrame(
        review_plan_rows
    )
    .merge(
        hr_analysis[
            [
                "representative_id",
                "job_title",
                "G",
            ]
        ],
        on="representative_id",
        how="left",
        validate="many_to_one",
    )
)

display(management_review_plan)


In [ ]:
# Apply progressive management feedback

feedback_target = y.copy()
feedback_predictions = (
    production_predictions_raw.copy()
)

feedback_stage_targets = {
    0: feedback_target.copy(),
}

feedback_stage_predictions = {
    0: feedback_predictions.copy(),
}

feedback_stage_metrics = []
feedback_action_logs = []

for cutoff in (4, 7, 17):

    cohort_ids = (
        feedback_cohorts[
            cutoff
        ]
    )

    desired_order = (
        feedback_desired_orders[
            cutoff
        ]
    )

    (
        feedback_target,
        feedback_predictions,
        action_log,
        stage_metrics,
    ) = (
        feedback.sequential_cohort_feedback(
            X=X,
            ids=ids,
            start_scores=feedback_target,
            previous_predictions=feedback_predictions,
            cohort_ids=cohort_ids,
            desired_order=desired_order,
            cutoff=cutoff,
            model_spec=model_spec,
        )
    )

    feedback_stage_targets[
        cutoff
    ] = feedback_target.copy()

    feedback_stage_predictions[
        cutoff
    ] = feedback_predictions.copy()

    feedback_stage_metrics.append(
        {
            "review_cutoff":
                cutoff,
            "review_fraction":
                cutoff / len(hr_analysis),
            **stage_metrics,
        }
    )

    for action in action_log:
        feedback_action_logs.append(
            {
                "review_cutoff":
                    cutoff,
                **action,
            }
        )

for cutoff in (4, 7, 17):
    assert np.allclose(
        np.sort(
            feedback_stage_targets[
                cutoff
            ]
        ),
        np.sort(y),
    )

feedback_fitted_summary = (
    pd.DataFrame(
        feedback_stage_metrics
    )
)

feedback_action_audit = (
    pd.DataFrame(
        feedback_action_logs
    )
)

display(
    feedback_fitted_summary[
        [
            "review_cutoff",
            "review_fraction",
            "total_actions",
            "effective_actions",
            "no_op_actions",
            "ndcg_stage0",
            "ndcg_final",
            "delta_ndcg_final",
            "ndcg_best",
            "delta_ndcg_best",
            "best_stage",
            "final_alpha",
        ]
    ]
)

print("Management action audit:")
display(feedback_action_audit)

feedback_target_comparison = (
    hr_analysis[
        [
            "representative_id",
            "job_title",
            "G",
        ]
    ]
    .rename(
        columns={
            "G":
                "G_pre_feedback"
        }
    )
)

for cutoff in (4, 7, 17):
    feedback_target_comparison[
        f"G_after_top_{cutoff}_review"
    ] = feedback_stage_targets[
        cutoff
    ]

display(feedback_target_comparison)


In [ ]:
# Test whether management feedback improves generalization

feedback_generalization_runs = {}

for cutoff in (4, 7, 17):

    (
        stage_oof_predictions,
        stage_fold_audit,
        stage_repeat_metrics,
        stage_oof_by_repeat,
        stage_alpha_tuning_trace,
    ) = (
        feedback.feedback_generalization_cv(
            X=X,
            target_scores=(
                feedback_stage_targets[
                    cutoff
                ]
            ),
            ids=ids,
            model_spec=model_spec,
        )
    )

    feedback_generalization_runs[
        cutoff
    ] = {
        "oof_predictions":
            stage_oof_predictions,
        "fold_audit":
            stage_fold_audit,
        "repeat_metrics":
            stage_repeat_metrics,
        "oof_by_repeat":
            stage_oof_by_repeat,
        "alpha_tuning_trace":
            stage_alpha_tuning_trace,
    }

feedback_primary_cutoffs = {
    4: 4,
    7: 7,
    17: 17,
}

stage_labels = {
    4: "10% review",
    7: "20% review",
    17: "50% review",
}

generalization_summary_rows = []
generalization_repeat_rows = []

for reviewed_k in (4, 7, 17):

    primary_k = (
        feedback_primary_cutoffs[
            reviewed_k
        ]
    )

    metric = (
        f"ndcg_at_{primary_k}"
    )

    baseline_repeat_values = (
        cv_repeat_metrics[
            metric
        ]
        .to_numpy(dtype=float)
    )

    adjusted_repeat_values = (
        feedback_generalization_runs[
            reviewed_k
        ][
            "repeat_metrics"
        ][
            metric
        ]
        .to_numpy(dtype=float)
    )

    baseline_mean = float(
        baseline_repeat_values.mean()
    )

    adjusted_mean = float(
        adjusted_repeat_values.mean()
    )

    generalization_summary_rows.append(
        {
            "review_stage":
                stage_labels[
                    reviewed_k
                ],
            "reviewed_n":
                reviewed_k,
            "review_fraction":
                reviewed_k
                / len(hr_analysis),
            "primary_metric":
                metric,
            "pre_feedback_mean":
                baseline_mean,
            "pre_feedback_sd":
                float(
                    baseline_repeat_values.std(
                        ddof=1
                    )
                ),
            "feedback_mean":
                adjusted_mean,
            "feedback_sd":
                float(
                    adjusted_repeat_values.std(
                        ddof=1
                    )
                ),
            "delta_vs_pre_feedback":
                adjusted_mean
                - baseline_mean,
        }
    )

    for repeat_index, (
        baseline_value,
        adjusted_value,
    ) in enumerate(
        zip(
            baseline_repeat_values,
            adjusted_repeat_values,
        ),
        start=1,
    ):
        generalization_repeat_rows.append(
            {
                "review_stage":
                    stage_labels[
                        reviewed_k
                    ],
                "reviewed_n":
                    reviewed_k,
                "primary_metric":
                    metric,
                "repeat":
                    repeat_index,
                "pre_feedback_ndcg":
                    baseline_value,
                "feedback_ndcg":
                    adjusted_value,
                "delta":
                    adjusted_value
                    - baseline_value,
            }
        )

feedback_generalization_summary = (
    pd.DataFrame(
        generalization_summary_rows
    )
)

feedback_generalization_repeats = (
    pd.DataFrame(
        generalization_repeat_rows
    )
)

display(
    feedback_generalization_summary
)

print(
    "Repeat-level matched comparisons:"
)
display(
    feedback_generalization_repeats
)


# Interpret management feedback and choose the production ranking

Management feedback changes the fitted ranking within the reviewed candidates, showing that the model can absorb a human preference signal when the analytical target is reassigned accordingly.

The stronger test is repeated nested cross-validation. At the matched 10%, 20% and 50% review depths, the feedback-adjusted targets do not improve the corresponding out-of-fold NDCG relative to the original pre-feedback model.

An improvement measured on the same candidates whose target assignments were changed demonstrates that the model can fit the feedback; it does not demonstrate that the resulting model ranks held-out candidates more effectively.

Management feedback is therefore retained as a governance and review mechanism rather than incorporated automatically into the production target. The final automated ranking remains the frozen pre-feedback PCA–Ridge ranking.


# Extend the governance audit to a top-25 review

The progressive 10%, 20% and 50% experiments test feedback and retraining within HR34. A separate top-25 experiment provides a broader **order-level governance audit**.

The previously approved strict management top-25 order is compared with the automated analytical ordering using NDCG@25. The existing $G$-score ladder remains the relevance reference; management changes only the ordering assigned to reviewed candidates.

The experiment is evaluated for the HR-relevant population and for the full set of 50 valid profiles. The full-50 version is a sensitivity audit only and does not replace HR34 as the production population.

Because retraining has already been tested in the progressive feedback experiment, this top-25 analysis is not another model-selection exercise. It quantifies agreement or disagreement between automated and management ordering at a deeper review depth.


In [ ]:
# Audit the management top-25 rerank

full50_ids = (
    clean_candidates[
        "representative_id"
    ]
    .astype(int)
    .to_numpy()
)

full50_analysis = (
    clean_candidates[
        [
            "representative_id",
            "job_title",
        ]
    ]
    .merge(
        hr_rule_audit[
            [
                "representative_id",
                "hr_class",
                "H",
            ]
        ],
        on="representative_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        semantic_filter_audit[
            [
                "representative_id",
                "W",
            ]
        ],
        on="representative_id",
        how="left",
        validate="one_to_one",
    )
)

full50_analysis["G"] = (
    full50_analysis["H"]
    + full50_analysis["W"]
) / 2.0

X50 = np.asarray(
    all50_embeddings,
    dtype=float,
)

y50 = (
    full50_analysis["G"]
    .to_numpy(dtype=float)
)

full50_alpha, full50_tuning_trace = (
    modeling.select_alpha(
        X=X50,
        y=y50,
        model_spec=model_spec,
    )
)

full50_model, full50_predictions_raw = (
    modeling.fit_full(
        X=X50,
        y=y50,
        alpha=full50_alpha,
        model_spec=model_spec,
    )
)

management_top25_order = [
    int(candidate_id)
    for candidate_id
    in management_source_top25_order
]


def audit_top25_rerank(
    population_name,
    population_ids,
    target_scores,
    model_scores,
    metadata,
    management_order,
):
    population_ids = np.asarray(
        population_ids,
        dtype=int,
    )
    target_scores = np.asarray(
        target_scores,
        dtype=float,
    )
    model_scores = np.asarray(
        model_scores,
        dtype=float,
    )

    available_ids = set(
        population_ids.tolist()
    )

    reviewed_ids = [
        int(candidate_id)
        for candidate_id
        in management_order
    ]

    if not set(reviewed_ids).issubset(
        available_ids
    ):
        missing = sorted(
            set(reviewed_ids)
            - available_ids
        )
        raise ValueError(
            "Recorded management top-25 "
            "contains IDs outside "
            f"{population_name}: {missing}"
        )

    if len(reviewed_ids) != 25:
        raise ValueError(
            "Recorded management review "
            "must contain exactly 25 IDs."
        )

    index = {
        int(candidate_id): row
        for row, candidate_id
        in enumerate(population_ids)
    }

    reviewed_target = np.array(
        [
            target_scores[
                index[candidate_id]
            ]
            for candidate_id
            in reviewed_ids
        ],
        dtype=float,
    )

    reviewed_model_scores = np.array(
        [
            model_scores[
                index[candidate_id]
            ]
            for candidate_id
            in reviewed_ids
        ],
        dtype=float,
    )

    management_score_ladder = np.sort(
        reviewed_target
    )[::-1]

    ndcg_before = ranking.ndcg_at(
        reviewed_target,
        reviewed_model_scores,
        k=25,
    )

    ndcg_after = ranking.ndcg_at(
        reviewed_target,
        management_score_ladder,
        k=25,
    )

    target_order = (
        ranking.deterministic_order(
            target_scores,
            population_ids,
        )
    )

    model_order = (
        ranking.deterministic_order(
            model_scores,
            population_ids,
        )
    )

    target_rank = {
        int(
            population_ids[row]
        ): rank
        for rank, row
        in enumerate(
            target_order,
            start=1,
        )
    }

    model_rank = {
        int(
            population_ids[row]
        ): rank
        for rank, row
        in enumerate(
            model_order,
            start=1,
        )
    }

    metadata_lookup = (
        metadata
        .set_index(
            "representative_id"
        )
    )

    audit_rows = []

    for management_position, (
        candidate_id,
        assigned_score,
    ) in enumerate(
        zip(
            reviewed_ids,
            management_score_ladder,
        ),
        start=1,
    ):
        audit_rows.append(
            {
                "population":
                    population_name,
                "representative_id":
                    candidate_id,
                "job_title":
                    metadata_lookup.loc[
                        candidate_id,
                        "job_title",
                    ],
                "original_G_rank":
                    target_rank[
                        candidate_id
                    ],
                "original_G_score":
                    float(
                        target_scores[
                            index[
                                candidate_id
                            ]
                        ]
                    ),
                "baseline_model_rank":
                    model_rank[
                        candidate_id
                    ],
                "management_position":
                    management_position,
                "management_assigned_score":
                    float(
                        assigned_score
                    ),
            }
        )

    result = {
        "population":
            population_name,
        "population_n":
            len(population_ids),
        "reviewed_n":
            len(reviewed_ids),
        "ndcg_at_25_before":
            float(ndcg_before),
        "ndcg_at_25_after":
            float(ndcg_after),
        "delta_ndcg_at_25":
            float(
                ndcg_after
                - ndcg_before
            ),
    }

    return (
        result,
        pd.DataFrame(audit_rows),
    )


hr34_top25_result, hr34_top25_audit = (
    audit_top25_rerank(
        population_name="HR34",
        population_ids=ids,
        target_scores=y,
        model_scores=(
            production_predictions_raw
        ),
        metadata=hr_analysis,
        management_order=(
            management_top25_order
        ),
    )
)

full50_top25_result, full50_top25_audit = (
    audit_top25_rerank(
        population_name="Full 50",
        population_ids=full50_ids,
        target_scores=y50,
        model_scores=(
            full50_predictions_raw
        ),
        metadata=full50_analysis,
        management_order=(
            management_top25_order
        ),
    )
)

top25_results = pd.DataFrame(
    [
        hr34_top25_result,
        full50_top25_result,
    ]
)

top25_rerank_audit = pd.concat(
    [
        hr34_top25_audit,
        full50_top25_audit,
    ],
    ignore_index=True,
)

print("Top-25 NDCG comparison:")
display(top25_results)

print(
    "Candidate-level top-25 rerank audit:"
)
display(top25_rerank_audit)


# Interpret the top-25 management rerank audit

The top-25 experiment is deliberately different from the progressive feedback-retraining analysis. Here, management ordering is evaluated directly against the existing analytical relevance target.

The original $G$ values remain the relevance reference. The candidate-level audit preserves each reviewed candidate's original $G$ score and rank alongside its automated model rank and management position, making disagreement inspectable rather than hidden.

HR34 remains the production population. The full-50 result is a sensitivity analysis showing the implications of relaxing occupational filtering.

The experiment therefore functions as an order-level governance diagnostic and does not replace the frozen production ranking.


In [ ]:
# Consolidate the principal analytical results

model_results_table = (
    cv_metric_summary.copy()
)

ridge_vs_w_table = (
    baseline_vs_ridge.copy()
)

feedback_results_table = (
    feedback_fitted_summary[
        [
            "review_cutoff",
            "review_fraction",
            "effective_actions",
            "no_op_actions",
            "ndcg_stage0",
            "ndcg_final",
            "delta_ndcg_final",
            "ndcg_best",
            "delta_ndcg_best",
            "best_stage",
        ]
    ]
    .merge(
        feedback_generalization_summary[
            [
                "reviewed_n",
                "primary_metric",
                "pre_feedback_mean",
                "pre_feedback_sd",
                "feedback_mean",
                "feedback_sd",
                "delta_vs_pre_feedback",
            ]
        ],
        left_on="review_cutoff",
        right_on="reviewed_n",
        how="left",
        validate="one_to_one",
    )
    .drop(columns="reviewed_n")
)

review_percentage_by_k = {
    int(k): int(round(100 * fraction))
    for fraction, k in config.FEEDBACK_DEPTH_TO_K.items()
}
feedback_results_table["review_percentage"] = (
    feedback_results_table["review_cutoff"].map(review_percentage_by_k)
)

top25_results_table = (
    top25_results.copy()
)

production_ranking_snapshot = (
    frozen_pre_feedback_ranking
    .head(17)
    .copy()
)

print(
    "Repeated nested-CV "
    "PCA-Ridge performance:"
)
display(model_results_table)

print(
    "PCA-Ridge versus "
    "deterministic W-only baseline:"
)
display(ridge_vs_w_table)

print(
    "Progressive management-feedback summary:"
)
display(feedback_results_table)

print(
    "Top-25 management rerank summary:"
)
display(top25_results_table)

print(
    "Frozen production ranking — top 17:"
)
display(production_ranking_snapshot)


In [ ]:
# Prepare the final reviewer-facing ranking table

final_ranking_table = (
    frozen_pre_feedback_ranking
    .copy()
)

final_ranking_table[
    "prediction_display"
] = np.clip(
    final_ranking_table[
        "prediction_raw"
    ],
    0.0,
    1.0,
)

final_ranking_table[
    "absolute_target_error"
] = np.abs(
    final_ranking_table["G"]
    - final_ranking_table[
        "prediction_raw"
    ]
)

final_ranking_table[
    "review_band"
] = np.select(
    [
        final_ranking_table[
            "ridge_rank"
        ] <= 4,
        final_ranking_table[
            "ridge_rank"
        ] <= 7,
        final_ranking_table[
            "ridge_rank"
        ] <= 17,
    ],
    [
        "Top 10% review band",
        "Top 20% review band",
        "Top 50% review band",
    ],
    default="Below 50% review band",
)

final_ranking_table = (
    final_ranking_table[
        [
            "ridge_rank",
            "representative_id",
            "job_title",
            "hr_class",
            "H",
            "W",
            "G",
            "prediction_raw",
            "prediction_display",
            "absolute_target_error",
            "review_band",
        ]
    ]
    .sort_values(
        "ridge_rank"
    )
    .reset_index(drop=True)
)

assert len(
    final_ranking_table
) == len(hr_analysis)

assert final_ranking_table[
    "representative_id"
].is_unique

print(
    "Final automated candidate ranking:"
)
display(final_ranking_table)


In [ ]:
# Figure F01 - Candidate population flow

population_flow = pd.DataFrame(
    {
        "stage": [
            "Raw rows",
            "Exact-title profiles",
            "Valid unique profiles",
            "HR modelling population",
        ],
        "candidates": [
            len(raw_df),
            len(exact_title_profiles),
            len(clean_candidates),
            len(hr_analysis),
        ],
    }
)

display(population_flow)

figures_dir = (
    ROOT
    / "outputs"
    / "figures"
)

figures_dir.mkdir(
    parents=True,
    exist_ok=True,
)

f01_path = (
    figures_dir
    / "F01_population_flow.png"
)

presentation.save_population_flow(
    path=f01_path,
    labels=population_flow["stage"],
    counts=population_flow[
        "candidates"
    ],
)

assert f01_path.exists()

display(
    Image(
        filename=str(f01_path)
    )
)


# Interpret Figure F01

Figure F01 shows four progressively smaller population stages: 104 raw source rows, 52 distinct exact-title profiles, 50 valid unique profiles, and 34 candidates in the final HR modelling population. The largest visible reduction occurs between the first two bars, where repeated profile text is consolidated from 104 source rows to 52 distinct exact-title profiles. The much smaller reduction from 52 to 50 reflects removal of the two invalid source records, after which normalized-title deduplication confirms the valid unique population. The final reduction from 50 to 34 occurs when the transparent occupational HR-relevance rules are applied.

This progression separates data-quality decisions from relevance modelling. Duplicate source text and invalid records are dealt with before occupational eligibility is considered, while semantic similarity is not used to decide which candidates enter the HR modelling population.

The resulting 34 candidates therefore represent a rule-defined HR-relevant population rather than a population selected by the later machine-learning model. This population is subsequently used to construct the analytical target, evaluate PCA–Ridge, and conduct the primary management-feedback experiments.


In [ ]:
# Figure F02 - Out-of-fold prediction versus target

repeat_keys = sorted(
    oof_by_repeat.keys()
)

oof_repeat_matrix = np.vstack(
    [
        np.asarray(
            oof_by_repeat[repeat],
            dtype=float,
        )
        for repeat in repeat_keys
    ]
)

assert oof_repeat_matrix.shape == (
    model_spec["outer_repeats"],
    len(hr_analysis),
)

mean_oof_prediction = (
    oof_repeat_matrix.mean(axis=0)
)

oof_prediction_table = (
    pd.DataFrame(
        {
            "representative_id":
                ids,
            "G":
                y,
            "mean_oof_prediction":
                mean_oof_prediction,
        }
    )
    .merge(
        hr_analysis[
            [
                "representative_id",
                "job_title",
            ]
        ],
        on="representative_id",
        how="left",
        validate="one_to_one",
    )
)

display(
    oof_prediction_table
    .sort_values(
        "G",
        ascending=False,
    )
)

f02_path = (
    figures_dir
    / "F02_oof_prediction_vs_target.png"
)

presentation.save_oof_scatter(
    path=f02_path,
    y=y,
    preds=mean_oof_prediction,
    ids=ids,
    highlight_id=4,
    highlight_label="ID 4 · adjacent HR",
)

assert f02_path.exists()

display(
    Image(
        filename=str(f02_path)
    )
)


# Interpret Figure F02

Figure F02 plots each candidate's analytical target score $G$ on the horizontal axis against that candidate's mean out-of-fold Ridge prediction on the vertical axis. The dashed diagonal represents perfect agreement between the target and the held-out prediction. Most candidates form a dense cluster near the upper-right portion of the figure and remain relatively close to this diagonal, indicating strong agreement for the predominantly direct-HR population.

One observation is clearly separated from the main cluster: candidate ID 4, the sole adjacent-HR candidate. Its analytical target is substantially lower because occupational relevance contributes $H=0.5$, whereas the candidate's embedding remains semantically similar to HR-related titles. Ridge therefore predicts a considerably higher relevance score than the constructed target for this case. This provides a visible example of information contained in the rule-based $H$ component that cannot necessarily be reconstructed from semantic title embeddings alone.

The remaining deviations around the diagonal show that PCA–Ridge approximates rather than exactly reproduces $G$. Importantly, every underlying prediction is out-of-fold: each candidate was excluded from the corresponding outer-fold training data when that prediction was generated.

The figure therefore provides evidence of held-out numerical agreement, while the repeated NDCG results remain the primary evaluation because the practical objective is candidate ranking rather than exact score reconstruction.


In [ ]:
# Figure F03 - Analytical target rank versus OOF Ridge rank

target_order = (
    ranking.deterministic_order(
        y,
        ids,
    )
)

oof_order = (
    ranking.deterministic_order(
        mean_oof_prediction,
        ids,
    )
)

target_ranks = np.empty(
    len(ids),
    dtype=int,
)

oof_ranks = np.empty(
    len(ids),
    dtype=int,
)

target_ranks[
    target_order
] = np.arange(
    1,
    len(ids) + 1,
)

oof_ranks[
    oof_order
] = np.arange(
    1,
    len(ids) + 1,
)

rank_plot_table = (
    pd.DataFrame(
        {
            "representative_id":
                ids,
            "target_rank":
                target_ranks,
            "mean_oof_ridge_rank":
                oof_ranks,
            "rank_difference":
                oof_ranks
                - target_ranks,
        }
    )
    .merge(
        hr_analysis[
            [
                "representative_id",
                "job_title",
            ]
        ],
        on="representative_id",
        validate="one_to_one",
    )
)

display(
    rank_plot_table
    .sort_values(
        "target_rank"
    )
)

f03_path = (
    figures_dir
    / "F03_target_vs_oof_rank.png"
)

presentation.save_rank_plot(
    path=f03_path,
    reference_ranks=target_ranks,
    predicted_ranks=oof_ranks,
)

assert f03_path.exists()

display(
    Image(
        filename=str(f03_path)
    )
)


# Interpret Figure F03

Figure F03 compares each candidate's analytical target rank on the horizontal axis with the rank obtained from the candidate's mean out-of-fold Ridge prediction on the vertical axis. Rank 1 represents the highest-ranked candidate on both axes. The dashed diagonal therefore represents exact agreement between the analytical target ordering and the out-of-fold model ordering.

Most observations remain reasonably close to the diagonal, particularly toward the strongest and weakest ends of the ranking. This shows that PCA–Ridge broadly preserves the candidate ordering encoded by the analytical target while still producing individual rank movements.

Points above the diagonal are ranked more highly by Ridge than by the analytical target, whereas points below the diagonal are ranked lower by Ridge. Some candidates move several positions, showing that strong overall ranking performance does not imply exact reproduction of every candidate's position.

This pattern is consistent with the metric results. NDCG remains very high because it places greater emphasis on preserving highly relevant candidates near the top of the ranking, while Spearman and Kendall measure agreement across the complete ordering and are therefore more sensitive to rank movements throughout the full population.

Figure F03 consequently complements the numerical ranking metrics by showing where ordering disagreement occurs rather than summarizing that disagreement in a single statistic.


In [ ]:
# Figure F04 - Management-feedback generalization

feedback_figure_table = (
    feedback_generalization_summary
    .copy()
    .sort_values(
        "reviewed_n"
    )
    .reset_index(drop=True)
)

display(
    feedback_figure_table[
        [
            "review_stage",
            "reviewed_n",
            "review_fraction",
            "primary_metric",
            "pre_feedback_mean",
            "feedback_mean",
            "delta_vs_pre_feedback",
        ]
    ]
)

f04_path = (
    figures_dir
    / "F04_feedback_generalization.png"
)

presentation.save_feedback(
    path=f04_path,
    df=feedback_figure_table,
    depth_col="review_fraction",
    before_col="pre_feedback_mean",
    after_col="feedback_mean",
)

assert f04_path.exists()

display(
    Image(
        filename=str(f04_path)
    )
)


# Interpret Figure F04

Figure F04 shows the change in held-out NDCG after incorporating management feedback at each of the three review depths. Each bar represents the feedback-adjusted repeated-CV result minus the corresponding pre-feedback repeated-CV result. The horizontal zero line therefore represents no change in held-out ranking performance.

All three bars fall below zero. At the 10% review stage, NDCG@4 changes from 0.989739 to 0.987483, a difference of -0.002256. At 20% review, NDCG@7 changes from 0.991032 to 0.985843, producing the largest observed decrease, -0.005188. At 50% review, NDCG@17 changes from 0.994684 to 0.992015, a difference of -0.002668.

The three bars correspond to different depth-matched NDCG cutoffs, so the purpose of the figure is not to compare the absolute NDCG level across review stages. Instead, it compares the direction and magnitude of the feedback effect relative to each stage's own pre-feedback reference.

Management-adjusted targets can be fitted by the model, but none of the three review depths produces an improvement in repeated held-out ranking performance. The available evidence therefore does not support replacing the frozen pre-feedback automated ranking with a model retrained to reproduce the management ordering.

Figure F05 examines the complementary question of whether greater management intervention at least produces progressively larger fitted gains before held-out generalization is considered.


In [ ]:
# Figure F05 - Management effort versus fitted gain

effort_figure_table = (
    feedback_fitted_summary[
        [
            "review_cutoff",
            "review_fraction",
            "effective_actions",
            "delta_ndcg_final",
        ]
    ]
    .copy()
    .sort_values(
        "review_cutoff"
    )
    .reset_index(drop=True)
)

display(effort_figure_table)

f05_path = (
    figures_dir
    / "F05_management_effort_vs_fitted_gain.png"
)

presentation.save_effort(
    path=f05_path,
    df=effort_figure_table,
    actions_col="effective_actions",
    gain_col="delta_ndcg_final",
)

assert f05_path.exists()

display(
    Image(
        filename=str(f05_path)
    )
)


# Interpret Figure F05

Figure F05 compares the number of effective management actions at each review depth with the corresponding change in fitted NDCG. The horizontal axis therefore represents the amount of management intervention that actually changed the frozen ordering, while the vertical axis shows the resulting fitted ranking gain on a $10^{-3}$ scale. The dotted vertical guides show the deviation of each fitted result from the zero-change reference without implying a continuous trend between the three experiments.

The three review experiments do not show a monotonic relationship between intervention effort and fitted improvement. At 10% review, three effective management actions correspond to a fitted NDCG change of approximately $-0.496 \times 10^{-3}$. At 20% review, only one effective action produces the largest positive fitted change, approximately $+0.535 \times 10^{-3}$. At 50% review, eight effective actions produce a smaller positive fitted change of approximately $+0.252 \times 10^{-3}$.

The magnitude of all three fitted changes is small, and increasing the number of management actions does not correspond to progressively greater fitted benefit. This indicates that the amount of intervention alone is not a useful proxy for how strongly the fitted ranking changes.

More importantly, fitted response and held-out generalization answer different questions. Figure F05 shows how strongly the model can respond to the management-adjusted targets within the fitted ranking, whereas Figure F04 shows whether those adjustments improve repeated held-out ranking performance. The latter remains negative at all three review depths.

Taken together, Figures F04 and F05 show that management feedback can alter the fitted ordering, but neither greater intervention nor those fitted changes provide evidence of improved generalization. This supports retaining the frozen pre-feedback automated ranking as the production result while preserving the management experiments as governance and sensitivity evidence.


In [ ]:
# Export final reviewer-facing outputs

outputs_dir = ROOT / "outputs"

outputs_dir.mkdir(
    parents=True,
    exist_ok=True,
)

final_ranking_path = (
    outputs_dir
    / "final_ranking.csv"
)

feedback_summary_path = (
    outputs_dir
    / "feedback_summary.csv"
)

run_log_path = (
    outputs_dir
    / "run.log"
)

run_manifest_path = (
    outputs_dir
    / "run_manifest.json"
)

final_ranking_table.to_csv(
    final_ranking_path,
    index=False,
)

feedback_export_rows = []

fitted_feedback_lookup = (
    feedback_fitted_summary
    .set_index(
        "review_cutoff"
    )
)

for _, row in (
    feedback_generalization_summary
    .iterrows()
):
    reviewed_n = int(
        row["reviewed_n"]
    )

    fitted_row = (
        fitted_feedback_lookup.loc[
            reviewed_n
        ]
    )

    feedback_export_rows.append(
        {
            "experiment":
                "progressive_feedback",
            "population":
                "HR34",
            "reviewed_n":
                reviewed_n,
            "review_fraction":
                float(
                    row[
                        "review_fraction"
                    ]
                ),
            "metric":
                row[
                    "primary_metric"
                ],
            "baseline_value":
                float(
                    row[
                        "pre_feedback_mean"
                    ]
                ),
            "post_feedback_value":
                float(
                    row[
                        "feedback_mean"
                    ]
                ),
            "delta":
                float(
                    row[
                        "delta_vs_pre_feedback"
                    ]
                ),
            "effective_actions":
                int(
                    fitted_row[
                        "effective_actions"
                    ]
                ),
            "fitted_delta_ndcg":
                float(
                    fitted_row[
                        "delta_ndcg_final"
                    ]
                ),
            "evaluation_scope":
                "repeated_nested_cv_generalization",
        }
    )

for _, row in (
    top25_results_table
    .iterrows()
):
    feedback_export_rows.append(
        {
            "experiment":
                "top25_management_rerank",
            "population":
                row["population"],
            "reviewed_n":
                int(
                    row["reviewed_n"]
                ),
            "review_fraction":
                float(
                    row["reviewed_n"]
                    / row["population_n"]
                ),
            "metric":
                "ndcg_at_25",
            "baseline_value":
                float(
                    row[
                        "ndcg_at_25_before"
                    ]
                ),
            "post_feedback_value":
                float(
                    row[
                        "ndcg_at_25_after"
                    ]
                ),
            "delta":
                float(
                    row[
                        "delta_ndcg_at_25"
                    ]
                ),
            "effective_actions":
                np.nan,
            "fitted_delta_ndcg":
                np.nan,
            "evaluation_scope":
                "direct_management_order_audit",
        }
    )

feedback_export_table = (
    pd.DataFrame(
        feedback_export_rows
    )
)

feedback_export_table.to_csv(
    feedback_summary_path,
    index=False,
)

run_log_text = "\n".join(
    [
        "Potential Talents final analytical run",
        f"raw_rows={len(raw_df)}",
        (
            "exact_title_profiles="
            f"{len(exact_title_profiles)}"
        ),
        (
            "valid_unique_profiles="
            f"{len(clean_candidates)}"
        ),
        (
            "hr_modeling_population="
            f"{len(hr_analysis)}"
        ),
        (
            "outer_repeats="
            f"{model_spec['outer_repeats']}"
        ),
        (
            "outer_splits="
            f"{model_spec['outer_splits']}"
        ),
        (
            "production_alpha="
            f"{production_alpha:.12g}"
        ),
        (
            "production_ranking="
            "pre_feedback_ridge"
        ),
        "",
    ]
)

io_utils.atomic_write_text(
    run_log_path,
    run_log_text,
)


def json_safe(value):
    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(value, dict):
        return {
            key: json_safe(item)
            for key, item
            in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    return value


source_hashes = {
    path.name:
        io_utils.sha256_file(path)
    for path in sorted(
        (ROOT / "src").glob(
            "*.py"
        )
    )
}

figure_paths = [
    f01_path,
    f02_path,
    f03_path,
    f04_path,
    f05_path,
]

artifact_paths = [
    final_ranking_path,
    feedback_summary_path,
    run_log_path,
    *figure_paths,
]

output_hashes = {
    str(
        path.relative_to(ROOT)
    ):
        io_utils.sha256_file(
            path
        )
    for path in artifact_paths
}

requirements_candidates = [
    ROOT / "requirements-lock.txt",
    ROOT / "requirements.txt",
]

requirements_hashes = {
    path.name:
        io_utils.sha256_file(path)
    for path
    in requirements_candidates
    if path.exists()
}

run_manifest = {
    "raw_data_file":
        str(
            raw_path.relative_to(ROOT)
        ),
    "raw_data_sha256":
        raw_sha256,
    "population_counts": {
        "raw_rows":
            int(len(raw_df)),
        "exact_title_profiles":
            int(
                len(
                    exact_title_profiles
                )
            ),
        "valid_unique_profiles":
            int(
                len(
                    clean_candidates
                )
            ),
        "hr_modeling_population":
            int(
                len(hr_analysis)
            ),
    },
    "model_spec":
        json_safe(model_spec),
    "production_alpha":
        float(production_alpha),
    "production_ranking":
        "frozen_pre_feedback_ridge",
    "source_sha256":
        source_hashes,
    "requirements_sha256":
        requirements_hashes,
    "output_sha256":
        output_hashes,
}

io_utils.atomic_write_json(
    run_manifest_path,
    run_manifest,
)

export_paths = [
    final_ranking_path,
    feedback_summary_path,
    run_log_path,
    run_manifest_path,
    *figure_paths,
]

export_summary = pd.DataFrame(
    {
        "artifact": [
            str(
                path.relative_to(ROOT)
            )
            for path in export_paths
        ],
        "exists": [
            path.exists()
            for path in export_paths
        ],
    }
)

display(export_summary)


In [ ]:
# Run final reproducibility and integrity checks

audit_checks = []


def record_check(
    code,
    condition,
    detail,
):
    validation.require(
        condition,
        code,
        detail,
    )

    audit_checks.append(
        {
            "check":
                code,
            "status":
                "PASS",
            "detail":
                detail,
        }
    )


record_check(
    "RAW_ROWS",
    len(raw_df) == 104,
    "Raw source reproduces 104 rows.",
)

record_check(
    "EXACT_TITLE_PROFILES",
    len(exact_title_profiles)
    == 52,
    (
        "Exact-title universe "
        "reproduces 52 profiles."
    ),
)

record_check(
    "VALID_UNIQUE_PROFILES",
    len(clean_candidates)
    == 50,
    (
        "Cleaning and deduplication "
        "reproduce 50 valid profiles."
    ),
)

record_check(
    "HR_MODELING_POPULATION",
    len(hr_analysis)
    == 34,
    (
        "Occupational filtering "
        "reproduces HR34."
    ),
)


record_check(
    "MODEL40_BINARY_PRESENT",
    (
        MODEL_BIN_PATH.exists()
        and MODEL_BIN_PATH.stat().st_size > 0
    ),
    (
        "The authoritative Model 40 binary "
        "is present and non-empty."
    ),
)

record_check(
    "MODEL40_METADATA",
    (
        int(model40_metadata["id"])
        == embeddings.EXPECTED_MODEL40_ID
        and int(
            model40_metadata[
                "vocabulary size"
            ]
        )
        == embeddings.EXPECTED_MODEL40_VOCAB
        and int(
            model40_metadata[
                "dimensions"
            ]
        )
        == embeddings.EXPECTED_DIM
    ),
    (
        "Model 40 metadata matches the "
        "locked model identity, vocabulary "
        "size and 100-dimensional representation."
    ),
)

record_check(
    "MODEL40_REQUIRED_VOCABULARY",
    (
        coverage_summary[
            "required_token_count"
        ]
        == len(required_vocabulary)
        and coverage_summary[
            "found_token_count"
        ]
        == len(vectors)
    ),
    (
        "Required-vocabulary coverage is "
        "recomputed directly from Model 40."
    ),
)

record_check(
    "TARGET_FORMULA",
    np.allclose(
        hr_analysis["G"],
        (
            hr_analysis["H"]
            + hr_analysis["W"]
        ) / 2,
    ),
    "G remains exactly (H + W) / 2.",
)

validation.require_finite(
    hr_analysis[
        [
            "H",
            "W",
            "G",
        ]
    ].to_numpy(),
    "FINITE_TARGET",
)

audit_checks.append(
    {
        "check":
            "FINITE_TARGET",
        "status":
            "PASS",
        "detail":
            (
                "H, W and G contain "
                "only finite values."
            ),
    }
)

expected_outer_fits = (
    model_spec["outer_splits"]
    * model_spec["outer_repeats"]
)

record_check(
    "OUTER_FITS",
    len(cv_fold_audit)
    == expected_outer_fits,
    (
        "Repeated nested CV "
        f"reproduces {expected_outer_fits} "
        "outer fits."
    ),
)

record_check(
    "COMPLETE_REPEATS",
    len(cv_repeat_metrics)
    == model_spec["outer_repeats"],
    (
        "All complete OOF repeats "
        "are retained."
    ),
)

expected_tuning_rows = (
    expected_outer_fits
    * len(
        model_spec[
            "alpha_grid"
        ]
    )
)

record_check(
    "TUNING_TRACE",
    len(alpha_tuning_trace)
    == expected_tuning_rows,
    (
        "Every outer fit retains "
        "every candidate Ridge alpha."
    ),
)

record_check(
    "ONE_ALPHA_PER_OUTER_FIT",
    int(
        alpha_tuning_trace[
            "selected"
        ].sum()
    )
    == expected_outer_fits,
    (
        "Exactly one Ridge alpha "
        "is selected per outer fit."
    ),
)

expected_production_ids = (
    ids[
        ranking.deterministic_order(
            production_predictions_raw,
            ids,
        )
    ]
)

actual_production_ids = (
    final_ranking_table[
        "representative_id"
    ].to_numpy()
)

record_check(
    "RAW_PREDICTION_AUTHORITY",
    np.array_equal(
        expected_production_ids,
        actual_production_ids,
    ),
    (
        "Final ranking is determined "
        "by raw predictions."
    ),
)

record_check(
    "PRE_FEEDBACK_PRODUCTION",
    np.array_equal(
        final_ranking_table[
            "representative_id"
        ].to_numpy(),
        frozen_pre_feedback_ranking[
            "representative_id"
        ].to_numpy(),
    ),
    (
        "Management feedback has not "
        "replaced the frozen ranking."
    ),
)

for cutoff in (4, 7, 17):
    record_check(
        f"FEEDBACK_MULTISET_{cutoff}",
        np.allclose(
            np.sort(
                feedback_stage_targets[
                    cutoff
                ]
            ),
            np.sort(y),
        ),
        (
            f"Feedback depth {cutoff} "
            "preserves the original "
            "G-score multiset."
        ),
    )

saved_manifest = json.loads(
    run_manifest_path.read_text(
        encoding="utf-8"
    )
)

record_check(
    "RAW_HASH",
    (
        saved_manifest[
            "raw_data_sha256"
        ]
        == io_utils.sha256_file(
            raw_path
        )
    ),
    (
        "Raw-data SHA-256 matches "
        "the run manifest."
    ),
)

for relative_path, expected_hash in (
    saved_manifest[
        "output_sha256"
    ].items()
):
    artifact_path = (
        ROOT / relative_path
    )

    record_check(
        (
            "OUTPUT_HASH_"
            + artifact_path.stem.upper()
        ),
        (
            artifact_path.exists()
            and io_utils.sha256_file(
                artifact_path
            )
            == expected_hash
        ),
        (
            f"{relative_path} exists "
            "and matches its recorded "
            "SHA-256."
        ),
    )

reproducibility_audit = (
    pd.DataFrame(
        audit_checks
    )
)

display(reproducibility_audit)

print(
    "All final reproducibility and "
    "integrity checks passed."
)


In [ ]:
# Build the comprehensive final NDCG summary table

ndcg_comparison_rows = []

for k in model_spec["ndcg_cutoffs"]:
    metric = f"ndcg_at_{k}"
    w_ndcg = ranking.ndcg_at(
        y_true=y,
        y_score=W_baseline_scores,
        k=k,
    )
    ridge_repeat_values = (
        cv_repeat_metrics[metric]
        .astype(float)
        .to_numpy()
    )
    ridge_mean = float(ridge_repeat_values.mean())
    ridge_sd = float(ridge_repeat_values.std(ddof=1))

    ndcg_comparison_rows.append(
        {
            "Experiment": "Automated ranking",
            "Population": "HR34",
            "Metric": f"NDCG@{k}",
            "Reference": "W-only",
            "Reference NDCG": float(w_ndcg),
            "Result": "PCA-Ridge",
            "Result NDCG": ridge_mean,
            "SD": ridge_sd,
            "Delta NDCG": ridge_mean - float(w_ndcg),
            "Effective actions": np.nan,
        }
    )

feedback_actions = (
    feedback_fitted_summary
    .set_index("review_cutoff")["effective_actions"]
    .to_dict()
)

for _, row in feedback_generalization_summary.iterrows():
    metric_name = str(row["primary_metric"])
    cutoff = int(metric_name.split("_")[-1])
    reviewed_n = int(row["reviewed_n"])
    review_pct = review_percentage_by_k[reviewed_n]

    ndcg_comparison_rows.append(
        {
            "Experiment": f"Management feedback - {review_pct}%",
            "Population": "HR34",
            "Metric": f"NDCG@{cutoff}",
            "Reference": "Pre-feedback Ridge",
            "Reference NDCG": float(row["pre_feedback_mean"]),
            "Result": "Feedback-adjusted Ridge",
            "Result NDCG": float(row["feedback_mean"]),
            "SD": float(row["feedback_sd"]),
            "Delta NDCG": float(row["delta_vs_pre_feedback"]),
            "Effective actions": int(feedback_actions[reviewed_n]),
        }
    )

for _, row in top25_results_table.iterrows():
    ndcg_comparison_rows.append(
        {
            "Experiment": "Top-25 management audit",
            "Population": row["population"],
            "Metric": "NDCG@25",
            "Reference": "Automated order",
            "Reference NDCG": float(row["ndcg_at_25_before"]),
            "Result": "Management order",
            "Result NDCG": float(row["ndcg_at_25_after"]),
            "SD": np.nan,
            "Delta NDCG": float(row["delta_ndcg_at_25"]),
            "Effective actions": np.nan,
        }
    )

ndcg_comparison_table = pd.DataFrame(ndcg_comparison_rows)

print("Comprehensive final NDCG summary:")
display(
    ndcg_comparison_table.style.format(
        {
            "Reference NDCG": "{:.6f}",
            "Result NDCG": "{:.6f}",
            "SD": "{:.6f}",
            "Delta NDCG": "{:+.6f}",
            "Effective actions": lambda value: (
                "—" if pd.isna(value) else f"{int(value)}"
            ),
        },
        na_rep="—",
    )
)


# Model performance observations

The combined evaluation results provide a consistent picture of model behaviour across ranking depth, baseline comparison, and management-feedback experiments.

NDCG is the primary performance measure because the practical objective is not simply to reproduce continuous target scores, but to place the most relevant candidates near the top of the ranking. Performance is therefore evaluated at several depths rather than only across the complete candidate list.

Across the HR34 population, PCA–Ridge achieves very high repeated out-of-fold NDCG at every evaluated cutoff. The strongest operational evidence is not confined to one value such as NDCG@10: performance remains consistently high at the shallow top-4 and top-7 levels, through the broader top-10 and top-17 ranges, and across the complete 34-candidate ordering. The relatively small standard deviations across the ten complete cross-validation repeats indicate that these ranking results are also stable with respect to the repeated fold assignments.

The deterministic $W$-only ranking performs at least as strongly as the learned Ridge model on the primary ranking metric. This result is structurally understandable. For 33 of the 34 HR candidates, $H=1$, which means

$$
G_i=\frac{1+W_i}{2}.
$$

Within that large subgroup, $G$ is therefore a monotonic transformation of $W$, so the deterministic semantic score already contains most of the ordering information present in the analytical target. The W-only comparison should consequently be interpreted as a simplicity benchmark rather than an independent competing model trained against an unrelated ground truth.

The Ridge model nevertheless reproduces the target ranking with strong out-of-fold performance while learning exclusively from the 100-dimensional Word2Vec candidate representation. Its value is therefore not that it dramatically outperforms $W$, but that it demonstrates that the analytical relevance structure can be recovered from the embedding representation within a leakage-controlled machine-learning pipeline.

The secondary regression metrics reinforce this interpretation. MAE and RMSE assess numerical agreement between predicted and target relevance, while Spearman and Kendall evaluate ordering across the complete population. These metrics are useful supporting diagnostics, but their values need not move identically with NDCG because they weight ranking errors differently. NDCG emphasizes mistakes near the top of the candidate list, where screening decisions are operationally most important.

The tuning-objective sensitivity analysis also shows that replacing inner-fold MSE tuning with direct NDCG tuning produces little material change in ranking performance or top-candidate membership. This supports retaining MSE as the production tuning objective: it provides a simple continuous optimization criterion without sacrificing meaningful ranking quality.

Management feedback produces a different result. The fitted experiments respond to the management-adjusted targets, but the amount of intervention does not translate into progressively greater fitted benefit: the 10%, 20%, and 50% review stages contain 3, 1, and 8 effective actions respectively, while their fitted NDCG changes remain small and non-monotonic. More importantly, the matched repeated-CV comparisons show held-out NDCG changes of -0.002256, -0.005188, and -0.002668 at the corresponding review depths. The distinction between fitted response and held-out generalization is therefore critical: management feedback changes the fitted ordering, but none of the three experiments improves out-of-fold ranking performance.

The top-25 rerank experiments provide an additional governance perspective. They measure agreement between the automated analytical ordering and the strict management ordering at a substantially deeper review depth, both within HR34 and in the full 50-profile sensitivity population. These results quantify disagreement but do not independently establish that either ordering represents superior future hiring outcomes.

Taken together, the evidence supports five pre-conclusion observations:

1. The automated PCA–Ridge pipeline produces consistently strong out-of-fold ranking performance across all evaluated NDCG depths.
2. Ranking stability across repeated cross-validation is high, with relatively limited variation between complete OOF repeats.
3. The deterministic semantic $W$ baseline remains extremely strong because it is structurally embedded in the construction of $G$.
4. Direct NDCG hyperparameter tuning does not materially outperform the simpler MSE-based production tuning strategy.
5. Management feedback changes the fitted ranking, but neither greater intervention nor the observed fitted changes translate into improved repeated out-of-fold ranking performance.

The performance evidence therefore favours the frozen pre-feedback analytical ranking as the reproducible automated output, while retaining management review as a separate human-governance layer rather than treating every manual reorder as new training truth.


# Limitations

This analysis provides a reproducible framework for candidate prioritization, but several limitations constrain how the results should be interpreted.

The final modelling population contains only 34 candidates. Repeated nested cross-validation reduces dependence on a single train/test split, but it cannot create information absent from a small dataset. Reported uncertainty therefore describes variation within this dataset rather than performance across the wider labour market.

The analytical target is constructed rather than externally observed. $G=(H+W)/2$ measures consistency with the project's stated HR-search objective rather than subsequent employee performance, interview success, retention, or another independent hiring outcome. This also explains why the deterministic $W$-only baseline is structurally strong.

Semantic evidence is based on job-title text and a pretrained Word2Vec representation. Titles are short, inconsistent descriptions of professional experience and may omit transferable skills, career intentions, seniority, or relevant experience contained elsewhere in a résumé.

The occupational filter intentionally favours precision. Explicit HR titles are retained, one defined adjacent case receives partial relevance, and candidates without occupational HR evidence are excluded from the production population. This reduces recruiter workload but creates a risk of excluding unconventional candidates whose suitability is not visible from the title alone.

The two search queries were derived from recurring language observed within the candidate data. The semantic filter stress test should therefore be interpreted as an internal consistency test rather than independent external validation.

Management feedback is based on one recorded ordering exercise over a limited candidate universe. The feedback experiments show how the pipeline responds to those preferences, but they do not establish that the management order represents a universally superior hiring standard.

Historical manual relevance grades are deliberately excluded from $H$, $W$, $G$, model training, and the final ranking. This avoids allowing coarse subjective labels with substantial ties to become an artificial ground truth.

Finally, the analysis does not use protected demographic attributes and does not constitute a fairness audit. A production recruitment system would require additional validation for disparate impact, accessibility, legal requirements, changing labour-market conditions, data drift, and downstream hiring outcomes before automated rankings were used for consequential employment decisions.


# Final conclusions

This project develops a reproducible candidate-ranking workflow designed to reduce manual screening when searching for people with human-resources experience or interest.

Data validation, invalid-record removal, normalized-title deduplication, and explicit occupational rules reduce the raw source to a transparent HR modelling population before machine learning is introduced.

Semantic relevance is represented using pretrained Word2Vec embeddings and two search queries derived from the language of the candidate population. Occupational relevance $H$ and semantic relevance $W$ are combined into

$$
G_i=\frac{H_i+W_i}{2}.
$$

A PCA–Ridge model then learns this target from the 100-dimensional candidate embeddings. Repeated nested cross-validation shows strong NDCG across multiple ranking depths while preserving held-out evaluation and fold-specific hyperparameter tuning.

The deterministic $W$-only baseline remains exceptionally strong because $W$ is structurally embedded in $G$ and occupational relevance is constant for almost all members of HR34. The learned model should therefore be viewed as a reproducible model of the analytical relevance structure rather than evidence that additional complexity necessarily outperforms the semantic score itself.

Management-feedback experiments provide a governance test. Ridge can fit management-adjusted orderings, but repeated nested cross-validation does not show improved generalization at the 10%, 20%, or 50% review depths. The separate top-25 audit likewise remains a diagnostic of disagreement between analytical and management ordering rather than a reason to redefine the production model automatically.

The final automated output is therefore the frozen pre-feedback PCA–Ridge ranking. Management review remains available as an explicit human-oversight layer for cases where additional business information, context or judgement justifies an override.

The practical value of the project is not only the ranking itself. The pipeline makes source cleaning, occupational eligibility, query construction, semantic coverage, target construction, hyperparameter tuning, out-of-fold evaluation, management intervention, final ranking and reproducibility checks traceable from one executable notebook.

The resulting system should therefore be understood as a transparent decision-support workflow rather than an autonomous hiring mechanism.
